In [34]:
# Task 2.1

import sqlite3
import csv

conn = sqlite3.connect("quiz.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS question")
cur.execute("DROP TABLE IF EXISTS scores")

cur.execute("""
CREATE TABLE IF NOT EXISTS question(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question_text TEXT NOT NULL,
    option_a TEXT NOT NULL,
    option_b TEXT NOT NULL,
    option_c TEXT NOT NULL,
    option_d TEXT NOT NULL,
    correct_option TEXT NOT NULL CHECK (correct_option IN ('A', 'B', 'C', 'D'))
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS scores(
    username TEXT PRIMARY KEY,
    score INTEGER NOT NULL
)
""")

conn.commit()

questions_csv = open("questions.csv", 'r')
questions = []

for line in csv.reader(questions_csv):
    questions.append(line)

questions_csv.close()

cur.executemany("""
INSERT INTO question(question_text, option_a, option_b, option_c, option_d, correct_option) VALUES (?, ?, ?, ?, ?, ?)
""", questions[1:])

conn.commit()
conn.close()

In [35]:
# Task 2.2

class Question:
    def __init__(self, question_text, option_a, option_b, option_c, option_d, correct_option):
        self.question_text = question_text
        self.option_a = option_a
        self.option_b = option_b
        self.option_c = option_c
        self.option_d = option_d
        self.correct_option = correct_option

    def to_str(self):
        return f"""{self.question_text}
A. {self.option_a}
B. {self.option_b}
C. {self.option_c}
D. {self.option_d}
Your answer (A/B/C/D):"""

    def check_answer(self, userchoice):
        return userchoice in ['A', 'B', 'C', 'D'] and self.correct_option == userchoice

In [36]:
# Task 2.3

from random import sample

def generate_5_questions():
    conn = sqlite3.connect("quiz.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM question")
    rows = cur.fetchall()
    conn.close()

    questions = sample(rows, k = 5)
    question_objects = []

    for q in questions:
        question_objects.append(Question(q[1], q[2], q[3], q[4], q[5], q[6]))

    return question_objects
    
question_objects = generate_5_questions()

for q in question_objects:
    print(q.to_str())

What is the capital of France?
A. Berlin
B. Madrid
C. Paris
D. Rome
Your answer (A/B/C/D):
What is the largest planet?
A. Earth
B. Mars
C. Jupiter
D. Saturn
Your answer (A/B/C/D):
What is the boiling point of water?
A. 90Â°C
B. 95Â°C
C. 100Â°C
D. 105Â°C
Your answer (A/B/C/D):
Who wrote 'Hamlet'?
A. Shakespeare
B. Dickens
C. Austen
D. Tolkien
Your answer (A/B/C/D):
What is the chemical symbol for water?
A. H2O
B. O2
C. CO2
D. NaCl
Your answer (A/B/C/D):


In [37]:
# Task 2.4

class QuizSession:
    def __init__(self, username):
        self.username = username
        self.score = 0

    def updateScoresDB(self):
        conn = sqlite3.connect("quiz.db")
        cur = conn.cursor()

        cur.execute("SELECT username FROM scores")
        rows = cur.fetchall()
        
        if (self.username, ) not in rows:
            cur.execute(f"INSERT INTO scores(username, score) VALUES ('{self.username}', {self.score})")
            conn.commit()
            return "Your score has been updated on quiz.db"

        else:
            cur.execute(f"SELECT score FROM scores WHERE scores.username = '{self.username}'")
            score = cur.fetchall()[0][0]

            if self.score > score:
                cur.execute(f"UPDATE scores SET score = {self.score} WHERE scores.username = '{self.username}'")
                conn.commit()
                return "Congratulations, you have achieved a new high score"
                
            else:
                return "You did not outperform your previous score."

        conn.close()    

In [38]:
# Task 2.5

def getLeaderBoard():
    conn = sqlite3.connect("quiz.db")
    cur = conn.cursor()

    cur.execute("SELECT * FROM scores ORDER BY scores.score DESC LIMIT 3")
    rows = cur.fetchall()
    conn.close()

    result = "Top 3 players\n==============="

    for player in rows:
        result += f"\n{player[0]}: {player[1]}pts"

    return result

In [43]:
# Task 2.6

import socket

server = socket.socket()
server.bind(("127.0.0.1", 12345))
server.listen()

new_socket, addr = server.accept()
print("Connected to:", addr)

new_socket.sendall(b"Enter your name")
username = new_socket.recv(1024).decode()

q = QuizSession(username)

question_objects = generate_5_questions()

score = 0

for question in question_objects:
    new_socket.sendall(question.to_str().encode())
    answer = new_socket.recv(1024).decode()

    if question.check_answer(answer):
        score += 1

q.score = score

new_socket.sendall(f"Your final score: {score}\n".encode())
new_socket.sendall(f"{q.updateScoresDB()}\n".encode())
new_socket.sendall(getLeaderBoard().encode())

new_socket.close()
server.close()

Connected to: ('127.0.0.1', 64394)
